In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display, Markdown

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)


In [ ]:
# load dataset

project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"
df = pd.read_csv(data_dir / "train.csv")

display(Markdown("### Dataset loaded"))
display(Markdown(f"The training dataset has been loaded from `{data_dir / 'train.csv'}`."))

display(pd.DataFrame({
    "metric": ["Rows", "Columns"],
    "value": [df.shape[0], df.shape[1]]
}))


In [ ]:
# helper function

def section(title, description=None):
    display(Markdown(f"## {title}"))
    if description:
        display(Markdown(description))


def subsection(title, description=None):
    display(Markdown(f"### {title}"))
    if description:
        display(Markdown(description))


def show_table(data):
    display(data)

In [ ]:
# dataset overview

section(
    "Dataset overview",
    "This section gives a quick overview of the dataset size and memory usage."
)

n_rows, n_cols = df.shape
total_cells = n_rows * n_cols

overview = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Total cells",
        "Memory usage MB"
    ],
    "value": [
        n_rows,
        n_cols,
        total_cells,
        round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    ]
})

show_table(overview)

In [ ]:
# column structure

section(
    "Column structure",
    "This section summarizes data types, missing values and number of unique values for each column."
)

dtype_summary = df.dtypes.value_counts().reset_index()
dtype_summary.columns = ["dtype", "count"]

subsection("Data type summary")
show_table(dtype_summary)

column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notnull().sum().values,
    "missing_count": df.isnull().sum().values,
    "missing_%": (df.isnull().mean().values * 100).round(2),
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": (df.nunique(dropna=True).values / len(df) * 100).round(2)
})

column_summary = column_summary.sort_values(
    by=["missing_%", "unique_%"],
    ascending=[False, False]
)

subsection("Column summary")
show_table(column_summary)

In [ ]:
# missing values

section(
    "Missing values",
    "This section identifies columns and rows affected by missing values. High missingness may require dropping, imputation or special handling."
)

missing_summary = pd.DataFrame({
    "metric": [
        "Total missing cells",
        "Total missing %",
        "Rows with at least one missing value",
        "Rows with at least one missing value %"
    ],
    "value": [
        df.isnull().sum().sum(),
        round(df.isnull().sum().sum() / total_cells * 100, 2),
        df.isnull().any(axis=1).sum(),
        round(df.isnull().any(axis=1).mean() * 100, 2)
    ]
})

show_table(missing_summary)

missing_df = column_summary[
    ["column", "missing_count", "missing_%"]
].sort_values("missing_%", ascending=False)

subsection("Columns sorted by missing values")
show_table(missing_df)

missing_groups = pd.DataFrame({
    "group": [
        "More than 90% missing",
        "More than 50% missing",
        "Between 10% and 50% missing",
        "No missing values"
    ],
    "columns": [
        missing_df[missing_df["missing_%"] > 90]["column"].tolist(),
        missing_df[missing_df["missing_%"] > 50]["column"].tolist(),
        missing_df[(missing_df["missing_%"] >= 10) & (missing_df["missing_%"] <= 50)]["column"].tolist(),
        missing_df[missing_df["missing_count"] == 0]["column"].tolist()
    ]
})

subsection("Missingness groups")
show_table(missing_groups)

In [ ]:
# target definition and distribution

section(
    "Target definition and distribution",
    "The target is 1 if the request was closed within 24 hours, and 0 otherwise. Missing, invalid, or negative closure times are assigned to class 0."
)

created = pd.to_datetime(df["Created Date"], errors="coerce")
closed = pd.to_datetime(df["Closed Date"], errors="coerce")

resolution_hours = (closed - created).dt.total_seconds() / 3600
valid_non_negative_resolution = resolution_hours.notna() & (resolution_hours >= 0)

target_24h = (
    valid_non_negative_resolution
    & (resolution_hours <= 24)
).astype(int)

target_quality = pd.DataFrame({
    "metric": [
        "Total rows",
        "Rows with missing Created Date",
        "Rows with missing Closed Date",
        "Rows with invalid resolution time",
        "Rows with negative resolution time",
        "Rows assigned target 1",
        "Rows assigned target 0"
    ],
    "value": [
        len(df),
        created.isna().sum(),
        closed.isna().sum(),
        resolution_hours.isna().sum(),
        (resolution_hours < 0).sum(),
        int(target_24h.sum()),
        int((target_24h == 0).sum())
    ]
})

subsection("Target quality")
show_table(target_quality)

target_distribution = target_24h.value_counts(dropna=False).sort_index().reset_index()
target_distribution.columns = ["target_24h", "count"]
target_distribution["percentage"] = (
    target_distribution["count"] / len(target_24h) * 100
).round(2)

subsection("Target distribution")
show_table(target_distribution)

target_distribution.set_index("target_24h")["count"].plot(
    kind="bar",
    figsize=(6, 4),
    title="Target distribution"
)

plt.xlabel("Closed within 24 hours")
plt.ylabel("Count")
plt.show()


In [ ]:
# main categorical features distribution

section(
    "Main categorical feature distributions",
    "This section shows the most frequent values for key categorical variables. This helps identify dominant categories and class imbalance inside features."
)

main_categorical_cols = [
    "Agency",
    "Problem (formerly Complaint Type)",
    "Borough"
]

for col in main_categorical_cols:
    if col in df.columns:
        subsection(col)

        distribution = df[col].value_counts(dropna=False).head(15).reset_index()
        distribution.columns = [col, "count"]
        distribution["percentage"] = (
            distribution["count"] / len(df) * 100
        ).round(2)

        show_table(distribution)

        distribution.set_index(col)["count"].plot(
            kind="bar",
            figsize=(10, 4),
            title=f"Top values: {col}"
        )

        plt.xlabel(col)
        plt.ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

In [ ]:
# temporal analysis

section(
    "Temporal analysis",
    "This section analyzes when complaints are created. Temporal patterns may be useful for feature engineering."
)

temporal_df = pd.DataFrame({
    "created_hour": created.dt.hour,
    "created_dayofweek": created.dt.dayofweek,
    "created_month": created.dt.month
})

day_mapping = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

temporal_df["created_day_name"] = temporal_df["created_dayofweek"].map(day_mapping)

subsection("Created hour distribution")

hour_distribution = temporal_df["created_hour"].value_counts(dropna=False).sort_index().reset_index()
hour_distribution.columns = ["created_hour", "count"]
hour_distribution["percentage"] = (
    hour_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(hour_distribution)

hour_distribution.set_index("created_hour")["count"].plot(
    kind="bar",
    figsize=(10, 4),
    title="Requests by creation hour"
)

plt.xlabel("Hour")
plt.ylabel("Count")
plt.show()


subsection("Created day of week distribution")

day_distribution = temporal_df["created_day_name"].value_counts(dropna=False).reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
).reset_index()

day_distribution.columns = ["created_day", "count"]
day_distribution["percentage"] = (
    day_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(day_distribution)

day_distribution.set_index("created_day")["count"].plot(
    kind="bar",
    figsize=(9, 4),
    title="Requests by day of week"
)

plt.xlabel("Day of week")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


subsection("Created month distribution")

month_distribution = temporal_df["created_month"].value_counts(dropna=False).sort_index().reset_index()
month_distribution.columns = ["created_month", "count"]
month_distribution["percentage"] = (
    month_distribution["count"] / len(temporal_df) * 100
).round(2)

show_table(month_distribution)

month_distribution.set_index("created_month")["count"].plot(
    kind="bar",
    figsize=(8, 4),
    title="Requests by creation month"
)

plt.xlabel("Month")
plt.ylabel("Count")
plt.show()

In [ ]:
# date consistency check

section(
    "Date consistency checks",
    "This section checks whether Created Date and Closed Date are valid and consistent."
)

date_checks = pd.DataFrame({
    "metric": [
        "Invalid Created Date values",
        "Invalid Closed Date values",
        "Missing Closed Date values",
        "Rows where Closed Date is before Created Date",
        "Negative resolution times",
        "Resolution times greater than 1 year"
    ],
    "value": [
        created.isna().sum() - df["Created Date"].isna().sum(),
        closed.isna().sum() - df["Closed Date"].isna().sum(),
        closed.isna().sum(),
        (closed < created).sum(),
        (resolution_hours < 0).sum(),
        (resolution_hours > 24 * 365).sum()
    ]
})

show_table(date_checks)

resolution_summary = resolution_hours.describe().reset_index()
resolution_summary.columns = ["statistic", "resolution_time_hours"]

subsection("Resolution time summary")
show_table(resolution_summary)

In [ ]:
# cardinality and feature difficulty

section(
    "Cardinality and feature difficulty",
    "This section identifies columns that may be difficult to use directly in modelling, such as ID-like or high-cardinality categorical variables."
)

cardinality_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "unique_count": df.nunique(dropna=True).values,
    "unique_%": (df.nunique(dropna=True).values / len(df) * 100).round(2),
    "missing_%": (df.isnull().mean().values * 100).round(2)
}).sort_values("unique_count", ascending=False)

show_table(cardinality_df)

feature_difficulty = pd.DataFrame({
    "group": [
        "ID-like or almost unique columns",
        "Low-cardinality columns",
        "High-missing columns",
        "Potentially difficult categorical columns"
    ],
    "columns": [
        cardinality_df[cardinality_df["unique_%"] > 90]["column"].tolist(),
        cardinality_df[cardinality_df["unique_count"] <= 10]["column"].tolist(),
        cardinality_df[cardinality_df["missing_%"] > 50]["column"].tolist(),
        cardinality_df[
            (cardinality_df["dtype"] == "object") &
            (cardinality_df["unique_count"] > 50) &
            (cardinality_df["unique_%"] <= 90)
        ]["column"].tolist()
    ]
})

subsection("Feature difficulty groups")
show_table(feature_difficulty)

In [ ]:
# geographic columns

section(
    "Geographic columns",
    "This section checks geographic variables and whether latitude and longitude can be converted to numeric values."
)

geo_cols = ["Latitude", "Longitude", "Location", "Incident Zip", "BBL"]
geo_info = []

for col in geo_cols:
    if col in df.columns:
        row = {
            "column": col,
            "dtype": df[col].dtype,
            "missing_%": round(df[col].isnull().mean() * 100, 2),
            "unique_values": df[col].nunique(dropna=True),
            "non_numeric_after_conversion": None,
            "min_after_conversion": None,
            "max_after_conversion": None
        }

        if col in ["Latitude", "Longitude"]:
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(",", ".", regex=False),
                errors="coerce"
            )

            row["non_numeric_after_conversion"] = converted.isnull().sum() - df[col].isnull().sum()
            row["min_after_conversion"] = converted.min()
            row["max_after_conversion"] = converted.max()

        geo_info.append(row)

geo_info_df = pd.DataFrame(geo_info)
show_table(geo_info_df)

In [ ]:
# potential data leakage

section(
    "Potential data leakage",
    "This section identifies columns that may contain information from after the complaint was created and should not be used as model features."
)

leakage_keywords = ["closed", "resolution", "status", "due"]

potential_leakage_cols = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

leakage_df = pd.DataFrame({
    "potential_leakage_column": potential_leakage_cols,
    "reason": [
        "May contain information unavailable at prediction time"
        for _ in potential_leakage_cols
    ]
})

show_table(leakage_df)

In [ ]:
# final summary

section(
    "Final diagnostic summary",
    "This section summarizes the main findings from the preliminary EDA."
)

final_summary = pd.DataFrame({
    "metric": [
        "Rows",
        "Columns",
        "Object columns",
        "Numeric columns",
        "Columns with more than 50% missing",
        "ID-like or almost unique columns",
        "Rows with missing Closed Date",
        "Rows assigned target 1",
        "Rows assigned target 0",
        "Majority class accuracy baseline"
    ],
    "value": [
        n_rows,
        n_cols,
        len(df.select_dtypes(include="object").columns),
        len(df.select_dtypes(include=np.number).columns),
        len(missing_df[missing_df["missing_%"] > 50]),
        len(cardinality_df[cardinality_df["unique_%"] > 90]),
        closed.isna().sum(),
        int(target_24h.sum()),
        int((target_24h == 0).sum()),
        round(target_24h.value_counts(normalize=True).max(), 4)
    ]
})

show_table(final_summary)

issues = pd.DataFrame({
    "main_issue": [
        "Many columns are stored as object dtype.",
        "Created Date and Closed Date need parsing before modelling.",
        "Closed Date is required to compute the target, but should not be used as a model feature.",
        "Missing or negative closure timestamps should be treated as not closed within 24 hours.",
        "Some columns have very high missingness.",
        "Some columns are ID-like or too high-cardinality for direct modelling.",
        "Categorical variables such as Agency, Complaint Type and Borough are likely important.",
        "Temporal features such as hour, day of week and month are likely useful.",
        "Geographic variables may require cleaning or conversion."
    ]
})

show_table(issues)
